In [1]:
from neuromeka import IndyDCP3 as RobotClient

In [2]:
robot = RobotClient(robot_ip="192.168.0.180")

In [3]:
robot.set_servo_all()

{'code': '0', 'msg': ''}

In [4]:
robot.get_robot_data()["op_state"]

5

In [4]:
robot.recover()

{'code': '0', 'msg': ''}

In [4]:
robot.get_robot_data()["q"]

[0.009303707,
 0.02499443,
 18.423412,
 0.00604384,
 0.007598877,
 -79.97816,
 -96.14019,
 -100.04042,
 -89.99857,
 -0.0125518795,
 -0.0057266233,
 -0.002483368,
 79.980896,
 96.12929,
 100.07029,
 90.020035,
 0.010588074,
 -0.001222229,
 0.0,
 0.0,
 0.0,
 0.0]

In [11]:
len(robot.get_robot_data()["pdot"])

18

In [ ]:
import inspect
import time

# IndyDCP GripperCommandType values (DH gripper uses gripper_type=2):
#   0 = AUTO_DETECT
#   1 = ACTIVATE
#   2 = DEACTIVATE
#   3 = SET_PVT
#
# Typical sequence after robot power-on:
#   gripper_auto_detect() -> gripper_activate() -> gripper_open() / gripper_close()
# If the gripper is already detected, skip auto-detect:
#   gripper_activate() -> gripper_open() / gripper_close()
GRIPPER_AUTO_DETECT = 0
GRIPPER_ACTIVATE = 1
GRIPPER_SET_PVT = 2
GRIPPER_DEACTIVATE = 3

DH_GRIPPER = 2

_GRIPPER_SUPPORTS_TOOL_INDEX = (
    "tool_index" in inspect.signature(robot.set_gripper_command).parameters
)

def _send_gripper_command(command, pvt_data, tool_index=0):
    """Send a command with both IndyDCP 3.4 and 3.5 SDKs."""
    kwargs = {
        "command": int(command),
        "gripper_type": DH_GRIPPER,
        "pvt_data": [int(value) for value in pvt_data],
    }
    if _GRIPPER_SUPPORTS_TOOL_INDEX:
        kwargs["tool_index"] = int(tool_index)
    elif int(tool_index) != 0:
        raise ValueError("This IndyDCP SDK only supports tool_index=0")
    return robot.set_gripper_command(**kwargs)

def gripper_auto_detect(tool_index=0):
    """Detect the DH gripper after a robot/controller restart."""
    return _send_gripper_command(
        GRIPPER_AUTO_DETECT, [1000, 50, 50, 0], tool_index)

def gripper_init(tool_index=0):
    """Backward-compatible alias for gripper_auto_detect()."""
    return gripper_auto_detect(tool_index)

def gripper_activate(tool_index=0):
    """Activate the detected DH gripper."""
    return _send_gripper_command(
        GRIPPER_ACTIVATE, [1000, 50, 50, 0], tool_index)

def gripper_deactivate(tool_index=0):
    """Deactivate the DH gripper."""
    return _send_gripper_command(
        GRIPPER_DEACTIVATE, [1000, 50, 50, 0], tool_index)

def gripper_move(position, tool_index=0, speed=50, force=50):
    """Set position, speed, and force for the DH gripper."""
    position = max(0, min(1000, int(round(position))))
    speed = max(1, min(100, int(round(speed))))
    force = max(20, min(100, int(round(force))))
    return _send_gripper_command(
        GRIPPER_SET_PVT, [position, speed, force, 0], tool_index)

def gripper_open(tool_index=0, speed=50, force=50):
    return gripper_move(
        position=1000, tool_index=tool_index, speed=speed, force=force)

def gripper_close(tool_index=0, speed=50, force=50):
    return gripper_move(
        position=0, tool_index=tool_index, speed=speed, force=force)

def get_gripper_state(tool_index=0):
    """Read status with both IndyDCP 3.4 and 3.5 SDKs."""
    getter = getattr(robot, "get_gripper_data_for", None)
    if callable(getter):
        return getter(int(tool_index))
    if int(tool_index) != 0:
        raise ValueError("This IndyDCP SDK only supports tool_index=0")
    return robot.get_gripper_data()

# Example sequences:
# After each servo-on:
#   gripper_auto_detect()  # command 0
#   gripper_open()       # command 3, [position=1000, speed=50, force=50, 0]
#   gripper_close()      # command 3, [position=0, speed=50, force=50, 0]
#   get_gripper_state()  # read current position

In [96]:
gripper_auto_detect(tool_index=1)

{}

In [93]:
gripper_open(tool_index=0)

{}

In [94]:
gripper_close(tool_index=0)

{}

In [28]:
robot.get_compliance_mode()

{'stiffness': [50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50,
  50],
 'enable': False}

In [19]:
robot.set_compliance_mode(enable=True, stiffness=[100]*22)

{'msg': 'SetSensorlessComplianceMode Success', 'code': '0'}

In [15]:
NOMINAL_HOME_POS = robot.get_robot_data()["q"]

In [16]:
import numpy as np

TARGET_POS = np.array(NOMINAL_HOME_POS)
TARGET_POS[:18] += 10

In [43]:
robot.set_compliance_mode(enable=True, stiffness=[100]*22)

VEL_RATIO = 10
ACC_RATIO = 10

robot.movej(jtarget=NOMINAL_HOME_POS, vel_ratio=VEL_RATIO, acc_ratio=ACC_RATIO)

{'code': '0', 'msg': ''}

In [101]:
init_pos_deg = [
    0.0,       # Joint_L0
    0.0,       # Joint_L1
    0.0,       # Joint_L2_U
    0.0,       # Joint_L3_U
    0.0,       # Joint_L2_L
    -79.985,   # Joint_L3_L
    -96.142,   # Joint_L4_L
    -100.096,  # Joint_L5_L
    -90.000,   # Joint_L6_L
    0.0,       # Joint_L7_L
    0.0,       # Joint_L8_L
    0.0,       # Joint_L2_R
    79.985,    # Joint_L3_R
    96.142,    # Joint_L4_R
    100.096,   # Joint_L5_R
    90.000,    # Joint_L6_R
    0.0,       # Joint_L7_R
    0.0,       # Joint_L8_R
    0.,
    0.,
    0.,
    0.
]

VEL_RATIO = 10
ACC_RATIO = 10

robot.movej(jtarget=init_pos_deg, vel_ratio=VEL_RATIO, acc_ratio=ACC_RATIO)

{'code': '0', 'msg': ''}

In [41]:
from neuromeka import StopCategory
robot.stop_motion(stop_category=StopCategory.CAT2)

{'msg': 'StopMotion Success', 'code': '0'}

In [4]:
# FK/IK test using the robot's current state. This cell does not move the robot.
import numpy as np

ARM_INDEX = 1  # 0: head, 1: left arm, 2: right arm
ACTIVE_JOINT_DOF = 18
ROBOT_JOINT_DOF = 22
TASK_DOF = 6

state = robot.get_robot_data()
current_q = np.asarray(state["q"], dtype=np.float64)
task_start = ARM_INDEX * TASK_DOF
current_task = np.asarray(
    state["p"][task_start:task_start + TASK_DOF],
    dtype=np.float64,
)

assert current_q.shape == (ROBOT_JOINT_DOF,), current_q.shape
assert current_task.shape == (TASK_DOF,), current_task.shape

fk_result = robot.forward_kin(
    jpos=current_q.tolist(),
    arm_index=ARM_INDEX,
)
ik_result = robot.inverse_kin(
    tpos=current_task.tolist(),
    init_jpos=current_q.tolist(),
    arm_index=ARM_INDEX,
)

print("current q22:", current_q.tolist())
print("current task pose:", current_task.tolist())
print("FK response:", fk_result.get("response"))
print("IK response:", ik_result.get("response"))

if str(fk_result.get("response", {}).get("code")) == "0":
    fk_task = np.asarray(fk_result["tpos"], dtype=np.float64)
    assert fk_task.shape == (TASK_DOF,), fk_task.shape
    print("FK task pose:", fk_task.tolist())
    print("FK - current task:", (fk_task - current_task).tolist())
else:
    print("FK failed:", fk_result)

if str(ik_result.get("response", {}).get("code")) == "0":
    ik_q = np.asarray(ik_result["jpos"], dtype=np.float64)
    if ik_q.shape == (ACTIVE_JOINT_DOF,):
        ik_q22 = np.concatenate([ik_q, np.zeros(4, dtype=np.float64)])
    elif ik_q.shape == (ROBOT_JOINT_DOF,):
        ik_q22 = ik_q.copy()
    else:
        raise ValueError(f"Unexpected IK joint shape: {ik_q.shape}")
    print("IK active joints:", ik_q.tolist())
    print("IK q22 with dummy zeros:", ik_q22.tolist())
    print("IK active - current active q:",
          (ik_q22[:ACTIVE_JOINT_DOF] - current_q[:ACTIVE_JOINT_DOF]).tolist())
else:
    print("IK failed:", ik_result)

current q22: [-0.01835927, 0.019959455, 0.030141018, 0.019280462, -138.69981, -25.197933, 76.68723, -99.8471, 77.274055, -37.65436, 4.343033, 137.05882, 25.108269, -73.13261, 102.04383, -73.95116, 37.882317, 0.01665802, 0.0, 0.0, 0.0, 0.0]
current task pose: [351.49078, 466.72107, 370.1288, 20.043913, -178.61748, 69.50214]
FK response: {'msg': 'ForwardKinematics Success', 'code': '0'}
IK response: {'msg': 'InverseKinematics Success', 'code': '0'}
FK task pose: [351.49078, 466.72107, 370.12878, 20.043911, -178.61748, 69.50214]
FK - current task: [0.0, 0.0, -2.0000000006348273e-05, -1.999999998503199e-06, 0.0, 0.0]
IK active joints: [-0.01835927, 0.019959455, 0.030141018, 0.019280462, -138.69981, -25.197933, 76.68723, -99.8471, 77.274055, -37.65436, 4.343033, 137.05882, 25.108269, -73.13261, 102.04383, -73.95116, 37.882317, 0.01665802]
IK q22 with dummy zeros: [-0.01835927, 0.019959455, 0.030141018, 0.019280462, -138.69981, -25.197933, 76.68723, -99.8471, 77.274055, -37.65436, 4.343033, 